# NYPD Complaint Data Historic - Comprehensive Exploratory Data Analysis

This notebook performs a detailed exploratory data analysis for the `NYPD_Complaint_Data_Historic.csv` dataset used by the NYC Crime Intelligence Dashboard project.

Primary goals:

- Understand schema, row volume, missingness, data types, and categorical cardinality.
- Validate incident dates, report dates, incident times, durations, and reporting lag.
- Audit geographic quality for borough, precinct, coordinate, transit, park, and housing fields.
- Analyze offense, law category, premise, jurisdiction, victim, and suspect fields.
- Produce trend, seasonality, borough, precinct, offense, hotspot, and anomaly-oriented summaries.
- Generate reusable outputs for the next pipeline phases:
  - `reports/data_quality_report.md`
  - `data/processed/schema_profile.json`
  - `data/processed/crime_weekly_area.parquet`
  - `data/processed/crime_monthly_area.parquet`
  - `data/processed/dashboard_summary.json`

Ethical scope:

- This notebook does not build a person-level crime prediction model.
- Demographic fields are profiled only for data quality, coverage, and fairness risk review.
- Recommended modeling features focus on time, location aggregates, offense categories, and historical counts.


## How to Run This Notebook

1. Start Jupyter from the repository or one of its subdirectories. The configuration cell locates the repository root automatically.

2. Place the raw source export at:

```text
data/raw/NYPD_Complaint_Data_Historic.csv
```

3. Run all cells from top to bottom. The full profiling cells scan a multi-gigabyte CSV and may take time.

4. If you need a faster first pass, set:

```python
RUN_FULL_PROFILE = False
SAMPLE_N = 250_000
```

5. For final project reporting, keep `RUN_FULL_PROFILE = True`.


In [ ]:
# Dependency setup
# This uses standard Python so it works consistently across Jupyter environments.
import importlib.util
import subprocess
import sys

REQUIRED_PACKAGES = {
    "duckdb": "duckdb",
    "polars": "polars",
    "pyarrow": "pyarrow",
    "plotly": "plotly",
    "missingno": "missingno",
    "folium": "folium",
    "scipy": "scipy",
    "statsmodels": "statsmodels",
    "sklearn": "scikit-learn",
    "tabulate": "tabulate",
}
missing_packages = [package for module, package in REQUIRED_PACKAGES.items() if importlib.util.find_spec(module) is None]

if missing_packages:
    print(f"Installing missing packages: {missing_packages}")
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *missing_packages])
else:
    print("All required packages are already installed.")


In [ ]:
from __future__ import annotations

import json
import math
import os
import re
import textwrap
import warnings
from collections import OrderedDict
from datetime import datetime, timezone
from pathlib import Path

import numpy as np
import pandas as pd

warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", 120)
pd.set_option("display.max_rows", 200)
pd.set_option("display.width", 180)
pd.set_option("display.float_format", lambda value: f"{value:,.4f}")

import duckdb
import plotly.express as px
import plotly.graph_objects as go
from IPython.display import Markdown, display


In [ ]:
# Project configuration
# Resolve all data and artifact locations from the repository root.
def find_project_root(start: Path) -> Path:
    resolved = start.resolve()
    for candidate in (resolved, *resolved.parents):
        if (candidate / "src").is_dir() and (candidate / "notebooks").is_dir():
            return candidate
    raise FileNotFoundError("Repository root not found from the current working directory.")


PROJECT_ROOT = find_project_root(Path.cwd())
RAW_CSV_PATH = PROJECT_ROOT / "data" / "raw" / "NYPD_Complaint_Data_Historic.csv"
PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"
REPORTS_DIR = PROJECT_ROOT / "reports"
FIGURES_DIR = REPORTS_DIR / "figures"
CACHE_DIR = PROJECT_ROOT / ".cache" / "eda"

for directory in [PROCESSED_DIR, REPORTS_DIR, FIGURES_DIR, CACHE_DIR]:
    directory.mkdir(parents=True, exist_ok=True)

RUN_FULL_PROFILE = True
SAMPLE_N = 500_000
TOP_N = 25
RANDOM_SEED = 42

NYC_LAT_MIN, NYC_LAT_MAX = 40.4774, 40.9176
NYC_LON_MIN, NYC_LON_MAX = -74.2591, -73.7004

print("Project root: repository root")
print(f"Raw CSV path: {RAW_CSV_PATH.relative_to(PROJECT_ROOT).as_posix()}")
print(f"Processed output directory: {PROCESSED_DIR.relative_to(PROJECT_ROOT).as_posix()}")
print(f"Reports output directory: {REPORTS_DIR.relative_to(PROJECT_ROOT).as_posix()}")


## Expected Source Columns

The dataset is expected to contain incident identifiers, dates and times, offense classification, law category, borough and precinct, coordinates, premise information, jurisdiction details, and victim/suspect demographic fields.

Demographic fields are sensitive and should not be used as initial forecasting features. They are included here only for completeness, coverage review, and fairness risk documentation.


In [ ]:
EXPECTED_COLUMN_GROUPS = OrderedDict({
    "identifier": ["CMPLNT_NUM"],
    "incident_datetime": ["CMPLNT_FR_DT", "CMPLNT_FR_TM", "CMPLNT_TO_DT", "CMPLNT_TO_TM", "RPT_DT"],
    "offense": ["KY_CD", "OFNS_DESC", "PD_CD", "PD_DESC", "CRM_ATPT_CPTD_CD", "LAW_CAT_CD"],
    "location_admin": ["BORO_NM", "ADDR_PCT_CD", "PATROL_BORO", "JURIS_DESC", "JURISDICTION_CODE"],
    "location_context": ["LOC_OF_OCCUR_DESC", "PREM_TYP_DESC", "PARKS_NM", "HADEVELOPT", "HOUSING_PSA", "TRANSIT_DISTRICT", "STATION_NAME"],
    "coordinates": ["X_COORD_CD", "Y_COORD_CD", "Latitude", "Longitude", "Lat_Lon"],
    "victim_demographics": ["VIC_AGE_GROUP", "VIC_RACE", "VIC_SEX"],
    "suspect_demographics": ["SUSP_AGE_GROUP", "SUSP_RACE", "SUSP_SEX"],
})

EXPECTED_COLUMNS = [column for group in EXPECTED_COLUMN_GROUPS.values() for column in group]
for group_name, columns in EXPECTED_COLUMN_GROUPS.items():
    print(f"{group_name:24s}: {', '.join(columns)}")


## File Existence and Raw Header Check

This cell validates that the expected CSV is available. It does not load the full file into memory.


In [ ]:
if not RAW_CSV_PATH.exists():
    raise FileNotFoundError(
        f"Raw CSV not found at {RAW_CSV_PATH}. Update PROJECT_ROOT or upload the file before continuing."
    )

file_size_gb = RAW_CSV_PATH.stat().st_size / (1024 ** 3)
header_preview = pd.read_csv(RAW_CSV_PATH, nrows=0)
raw_columns = list(header_preview.columns)

missing_expected = sorted(set(EXPECTED_COLUMNS) - set(raw_columns))
unexpected_columns = sorted(set(raw_columns) - set(EXPECTED_COLUMNS))

print(f"File size: {file_size_gb:,.2f} GB")
print(f"Column count: {len(raw_columns)}")
print(f"Missing expected columns: {missing_expected if missing_expected else 'None'}")
print(f"Unexpected columns: {unexpected_columns if unexpected_columns else 'None'}")
print("\nRaw columns:")
print(raw_columns)


## DuckDB Connection and SQL Helpers

The full CSV is read through DuckDB with all columns initially treated as strings. This avoids accidental type inference problems in fields with mixed values, placeholder nulls, and inconsistent formatting.


In [ ]:
def sql_string(value: str | Path) -> str:
    return "'" + str(value).replace("'", "''") + "'"


def ident(column: str) -> str:
    return '"' + column.replace('"', '""') + '"'


NULL_STRINGS = ["", "(null)", "NULL", "null", "NaN", "nan", "N/A", "n/a"]
NULL_SQL_LIST = "[" + ", ".join(sql_string(value) for value in NULL_STRINGS) + "]"
CSV_EXPR = f"""
read_csv_auto(
    {sql_string(RAW_CSV_PATH)},
    header=true,
    all_varchar=true,
    ignore_errors=true,
    nullstr={NULL_SQL_LIST}
)
""".strip()

con = duckdb.connect(database=str(CACHE_DIR / "eda.duckdb"))
con.execute("PRAGMA threads=4")
con.execute(f"CREATE OR REPLACE VIEW complaints_raw AS SELECT * FROM {CSV_EXPR}")


def q(sql: str, params: dict | None = None) -> pd.DataFrame:
    if params is None:
        return con.execute(sql).df()
    return con.execute(sql, params).df()


def show_df(df: pd.DataFrame, rows: int = 20) -> None:
    display(df.head(rows))

print("DuckDB view created: complaints_raw")


## Raw Row Count

The exact row count is the first full scan. The notebook deliberately does not display raw event rows.


In [ ]:
if RUN_FULL_PROFILE:
    row_count = int(q("SELECT COUNT(*) AS row_count FROM complaints_raw").loc[0, "row_count"])
else:
    row_count = np.nan

print(f"Full row count: {row_count:,}" if not pd.isna(row_count) else "Full row count skipped.")


## Normalized Analysis View

This view keeps original columns and adds parsed dates, parsed timestamps, numeric coordinates, numeric codes, and reusable quality flags.


In [ ]:
con.execute(f"""
CREATE OR REPLACE VIEW complaints_enriched AS
SELECT
    *,
    try_strptime(CMPLNT_FR_DT, '%m/%d/%Y')::DATE AS complaint_from_date,
    try_strptime(CMPLNT_TO_DT, '%m/%d/%Y')::DATE AS complaint_to_date,
    try_strptime(RPT_DT, '%m/%d/%Y')::DATE AS report_date,
    try_strptime(CMPLNT_FR_DT || ' ' || CMPLNT_FR_TM, '%m/%d/%Y %H:%M:%S') AS complaint_from_ts,
    try_strptime(CMPLNT_TO_DT || ' ' || CMPLNT_TO_TM, '%m/%d/%Y %H:%M:%S') AS complaint_to_ts,
    try_cast(ADDR_PCT_CD AS INTEGER) AS precinct_num,
    try_cast(KY_CD AS INTEGER) AS ky_cd_num,
    try_cast(PD_CD AS INTEGER) AS pd_cd_num,
    try_cast(JURISDICTION_CODE AS INTEGER) AS jurisdiction_code_num,
    try_cast(X_COORD_CD AS DOUBLE) AS x_coord_num,
    try_cast(Y_COORD_CD AS DOUBLE) AS y_coord_num,
    try_cast(Latitude AS DOUBLE) AS latitude_num,
    try_cast(Longitude AS DOUBLE) AS longitude_num,
    CASE
        WHEN try_cast(Latitude AS DOUBLE) BETWEEN {NYC_LAT_MIN} AND {NYC_LAT_MAX}
         AND try_cast(Longitude AS DOUBLE) BETWEEN {NYC_LON_MIN} AND {NYC_LON_MAX}
        THEN true ELSE false
    END AS has_valid_nyc_coordinates,
    CASE
        WHEN CMPLNT_FR_TM IS NULL THEN NULL
        WHEN regexp_matches(CMPLNT_FR_TM, '^(?:[01][0-9]|2[0-3]):[0-5][0-9]:[0-5][0-9]$') THEN true
        ELSE false
    END AS has_valid_from_time,
    CASE
        WHEN CMPLNT_TO_TM IS NULL THEN NULL
        WHEN regexp_matches(CMPLNT_TO_TM, '^(?:[01][0-9]|2[0-3]):[0-5][0-9]:[0-5][0-9]$') THEN true
        ELSE false
    END AS has_valid_to_time,
    CASE
        WHEN try_strptime(CMPLNT_FR_DT, '%m/%d/%Y')::DATE IS NULL THEN true ELSE false
    END AS has_invalid_from_date,
    CASE
        WHEN try_strptime(RPT_DT, '%m/%d/%Y')::DATE IS NULL THEN true ELSE false
    END AS has_invalid_report_date
FROM complaints_raw
""")
print("DuckDB view created: complaints_enriched")


## Analysis Sample

A reproducible reservoir sample is used for visualizations and heavier statistical checks that do not require a full scan. Full-data aggregates are still used where exact counts matter.


In [ ]:
def load_sample(sample_n: int = SAMPLE_N) -> pd.DataFrame:
    try:
        sampled = q(f"""
            SELECT *
            FROM complaints_enriched
            USING SAMPLE reservoir({sample_n} ROWS) REPEATABLE({RANDOM_SEED})
        """)
    except Exception as exc:
        print(f"DuckDB reservoir sample failed, falling back to first {sample_n:,} rows. Reason: {exc}")
        sampled = pd.read_csv(
            RAW_CSV_PATH,
            nrows=sample_n,
            dtype=str,
            na_values=NULL_STRINGS,
            keep_default_na=True,
            low_memory=False,
        )
    return sampled

sample_df = load_sample(SAMPLE_N)
print(f"Analysis sample rows: {len(sample_df):,}")
print(f"Analysis sample columns: {len(sample_df.columns):,}")


# 1. Schema, Missingness, and Cardinality

This section profiles every column using aggregate-safe row counts, missingness, presence percentages, and approximate cardinality. Lexical extrema and sample values are not published because they can reproduce event identifiers, exact coordinates, or location examples.


In [ ]:
def build_column_profile(columns: list[str]) -> pd.DataFrame:
    total_sql = "COUNT(*)" if RUN_FULL_PROFILE else str(len(sample_df))
    parts = []
    for column in columns:
        col = ident(column)
        parts.append(f"""
            SELECT
                {sql_string(column)} AS column_name,
                {total_sql} AS row_count,
                SUM(CASE WHEN {col} IS NULL THEN 1 ELSE 0 END) AS missing_count,
                100.0 * SUM(CASE WHEN {col} IS NULL THEN 1 ELSE 0 END) / NULLIF({total_sql}, 0) AS missing_pct,
                approx_count_distinct({col}) AS approx_unique_count,
                MIN({col}) AS min_lexical_value,
                MAX({col}) AS max_lexical_value
            FROM complaints_raw
        """)
    profile = q("\nUNION ALL\n".join(parts))
    profile["present_pct"] = 100 - profile["missing_pct"]
    profile = profile.sort_values(["missing_pct", "approx_unique_count"], ascending=[False, False]).reset_index(drop=True)
    return profile

column_profile = build_column_profile(raw_columns) if RUN_FULL_PROFILE else pd.DataFrame({
    "column_name": sample_df.columns,
    "row_count": len(sample_df),
    "missing_count": sample_df.isna().sum().values,
    "missing_pct": (sample_df.isna().mean() * 100).values,
    "approx_unique_count": [sample_df[column].nunique(dropna=True) for column in sample_df.columns],
})

if "present_pct" not in column_profile.columns:
    column_profile["present_pct"] = 100 - column_profile["missing_pct"]

PUBLIC_COLUMN_PROFILE_COLUMNS = ["column_name", "missing_pct", "approx_unique_count"]
SCHEMA_COLUMN_PROFILE_COLUMNS = [
    "column_name", "row_count", "missing_count", "missing_pct", "present_pct", "approx_unique_count"
]
column_profile_public = column_profile[PUBLIC_COLUMN_PROFILE_COLUMNS].copy()
column_profile_schema = column_profile[SCHEMA_COLUMN_PROFILE_COLUMNS].copy()
show_df(column_profile_public, 50)


In [ ]:
fig = px.bar(
    column_profile.sort_values("missing_pct", ascending=True),
    x="missing_pct",
    y="column_name",
    orientation="h",
    title="Missing Value Percentage by Column",
    labels={"missing_pct": "Missing (%)", "column_name": "Column"},
    height=max(600, 20 * len(column_profile)),
)
fig.update_layout(yaxis={"categoryorder": "total ascending"})
fig.show()


In [ ]:
fig = px.bar(
    column_profile.sort_values("approx_unique_count", ascending=False),
    x="column_name",
    y="approx_unique_count",
    title="Approximate Unique Count by Column",
    labels={"approx_unique_count": "Approximate unique values", "column_name": "Column"},
    height=550,
)
fig.update_layout(xaxis_tickangle=-45)
fig.show()


In [ ]:
def top_values(column: str, limit: int = TOP_N) -> pd.DataFrame:
    col = ident(column)
    return q(f"""
        SELECT
            {col} AS value,
            COUNT(*) AS row_count,
            100.0 * COUNT(*) / SUM(COUNT(*)) OVER () AS pct
        FROM complaints_raw
        GROUP BY 1
        ORDER BY row_count DESC
        LIMIT {limit}
    """)

KEY_PROFILE_COLUMNS = [
    "BORO_NM", "ADDR_PCT_CD", "PATROL_BORO", "LAW_CAT_CD", "OFNS_DESC", "PD_DESC",
    "CRM_ATPT_CPTD_CD", "PREM_TYP_DESC", "LOC_OF_OCCUR_DESC", "JURIS_DESC",
    "VIC_AGE_GROUP", "VIC_RACE", "VIC_SEX", "SUSP_AGE_GROUP", "SUSP_RACE", "SUSP_SEX",
]

for column in [col for col in KEY_PROFILE_COLUMNS if col in raw_columns]:
    display(Markdown(f"### Top values: `{column}`"))
    show_df(top_values(column, TOP_N), TOP_N)


## Missingness by Column Group

Grouping fields by business meaning makes the data quality issues easier to prioritize.


In [ ]:
profile_by_group = []
for group_name, columns in EXPECTED_COLUMN_GROUPS.items():
    group_cols = [column for column in columns if column in set(column_profile["column_name"])]
    subset = column_profile[column_profile["column_name"].isin(group_cols)].copy()
    if subset.empty:
        continue
    profile_by_group.append({
        "group": group_name,
        "column_count": len(subset),
        "avg_missing_pct": subset["missing_pct"].mean(),
        "max_missing_pct": subset["missing_pct"].max(),
        "highest_missing_column": subset.sort_values("missing_pct", ascending=False).iloc[0]["column_name"],
    })

group_profile = pd.DataFrame(profile_by_group).sort_values("avg_missing_pct", ascending=False)
show_df(group_profile, 30)

fig = px.bar(
    group_profile,
    x="group",
    y="avg_missing_pct",
    color="max_missing_pct",
    title="Average Missingness by Column Group",
    labels={"avg_missing_pct": "Average missing (%)", "group": "Column group", "max_missing_pct": "Max missing (%)"},
    height=450,
)
fig.update_layout(xaxis_tickangle=-30)
fig.show()


# 2. Date, Time, Duration, and Reporting Lag

Incident timing is the foundation for trend analysis, forecasting, anomaly detection, and dashboard filters. This section checks date ranges, invalid dates, invalid times, negative durations, long durations, and reporting lag.


In [ ]:
date_quality = q("""
SELECT
    COUNT(*) AS row_count,
    SUM(CASE WHEN complaint_from_date IS NULL THEN 1 ELSE 0 END) AS missing_or_invalid_from_date,
    SUM(CASE WHEN complaint_to_date IS NULL THEN 1 ELSE 0 END) AS missing_or_invalid_to_date,
    SUM(CASE WHEN report_date IS NULL THEN 1 ELSE 0 END) AS missing_or_invalid_report_date,
    MIN(complaint_from_date) AS min_from_date,
    MAX(complaint_from_date) AS max_from_date,
    MIN(complaint_to_date) AS min_to_date,
    MAX(complaint_to_date) AS max_to_date,
    MIN(report_date) AS min_report_date,
    MAX(report_date) AS max_report_date,
    SUM(CASE WHEN complaint_from_date > current_date THEN 1 ELSE 0 END) AS future_from_dates,
    SUM(CASE WHEN report_date > current_date THEN 1 ELSE 0 END) AS future_report_dates,
    SUM(CASE WHEN complaint_to_date IS NOT NULL AND complaint_from_date IS NOT NULL AND complaint_to_date < complaint_from_date THEN 1 ELSE 0 END) AS to_date_before_from_date
FROM complaints_enriched
""")
show_df(date_quality)


In [ ]:
time_quality = q("""
SELECT
    COUNT(*) AS row_count,
    SUM(CASE WHEN CMPLNT_FR_TM IS NULL THEN 1 ELSE 0 END) AS missing_from_time,
    SUM(CASE WHEN has_valid_from_time = false THEN 1 ELSE 0 END) AS invalid_from_time,
    SUM(CASE WHEN CMPLNT_TO_TM IS NULL THEN 1 ELSE 0 END) AS missing_to_time,
    SUM(CASE WHEN has_valid_to_time = false THEN 1 ELSE 0 END) AS invalid_to_time,
    SUM(CASE WHEN complaint_from_ts IS NULL THEN 1 ELSE 0 END) AS missing_or_invalid_from_timestamp,
    SUM(CASE WHEN complaint_to_ts IS NULL THEN 1 ELSE 0 END) AS missing_or_invalid_to_timestamp
FROM complaints_enriched
""")
show_df(time_quality)


In [ ]:
yearly_counts = q("""
SELECT
    EXTRACT(year FROM complaint_from_date)::INTEGER AS year,
    COUNT(*) AS complaint_count
FROM complaints_enriched
WHERE complaint_from_date IS NOT NULL
GROUP BY 1
ORDER BY 1
""")
yearly_counts["pct_change_yoy"] = yearly_counts["complaint_count"].pct_change() * 100
show_df(yearly_counts, 100)

fig = px.line(
    yearly_counts,
    x="year",
    y="complaint_count",
    markers=True,
    title="Annual Complaint Volume by Incident Start Date",
    labels={"complaint_count": "Complaints", "year": "Year"},
)
fig.show()


In [ ]:
monthly_counts = q("""
SELECT
    date_trunc('month', complaint_from_date)::DATE AS month_start,
    COUNT(*) AS complaint_count
FROM complaints_enriched
WHERE complaint_from_date IS NOT NULL
GROUP BY 1
ORDER BY 1
""")
monthly_counts["rolling_12_month_avg"] = monthly_counts["complaint_count"].rolling(12, min_periods=3).mean()
show_df(monthly_counts, 20)

fig = go.Figure()
fig.add_trace(go.Scatter(x=monthly_counts["month_start"], y=monthly_counts["complaint_count"], mode="lines", name="Monthly complaints"))
fig.add_trace(go.Scatter(x=monthly_counts["month_start"], y=monthly_counts["rolling_12_month_avg"], mode="lines", name="12-month rolling average"))
fig.update_layout(title="Monthly Complaint Trend", xaxis_title="Month", yaxis_title="Complaints", height=500)
fig.show()


In [ ]:
weekly_counts = q("""
SELECT
    date_trunc('week', complaint_from_date)::DATE AS week_start,
    COUNT(*) AS complaint_count
FROM complaints_enriched
WHERE complaint_from_date IS NOT NULL
GROUP BY 1
ORDER BY 1
""")
weekly_counts["rolling_8_week_avg"] = weekly_counts["complaint_count"].rolling(8, min_periods=3).mean()
show_df(weekly_counts, 20)

fig = go.Figure()
fig.add_trace(go.Scatter(x=weekly_counts["week_start"], y=weekly_counts["complaint_count"], mode="lines", name="Weekly complaints"))
fig.add_trace(go.Scatter(x=weekly_counts["week_start"], y=weekly_counts["rolling_8_week_avg"], mode="lines", name="8-week rolling average"))
fig.update_layout(title="Weekly Complaint Trend", xaxis_title="Week", yaxis_title="Complaints", height=500)
fig.show()


In [ ]:
month_year_matrix = monthly_counts.copy()
month_year_matrix["year"] = pd.to_datetime(month_year_matrix["month_start"]).dt.year
month_year_matrix["month"] = pd.to_datetime(month_year_matrix["month_start"]).dt.month
monthly_pivot = month_year_matrix.pivot(index="year", columns="month", values="complaint_count").sort_index()

fig = px.imshow(
    monthly_pivot,
    aspect="auto",
    color_continuous_scale="Viridis",
    title="Calendar Heatmap: Monthly Complaint Volume by Year",
    labels={"x": "Month", "y": "Year", "color": "Complaints"},
)
fig.update_xaxes(tickmode="array", tickvals=list(range(12)), ticktext=[str(i) for i in range(1, 13)])
fig.show()


In [ ]:
hourly_counts = q("""
SELECT
    EXTRACT(hour FROM complaint_from_ts)::INTEGER AS incident_hour,
    COUNT(*) AS complaint_count
FROM complaints_enriched
WHERE complaint_from_ts IS NOT NULL
GROUP BY 1
ORDER BY 1
""")

fig = px.bar(
    hourly_counts,
    x="incident_hour",
    y="complaint_count",
    title="Complaint Volume by Incident Hour",
    labels={"incident_hour": "Hour of day", "complaint_count": "Complaints"},
)
fig.show()

weekday_counts = q("""
SELECT
    strftime(complaint_from_date, '%w')::INTEGER AS weekday_num,
    strftime(complaint_from_date, '%A') AS weekday_name,
    COUNT(*) AS complaint_count
FROM complaints_enriched
WHERE complaint_from_date IS NOT NULL
GROUP BY 1, 2
ORDER BY 1
""")
show_df(weekday_counts)

fig = px.bar(
    weekday_counts,
    x="weekday_name",
    y="complaint_count",
    title="Complaint Volume by Day of Week",
    labels={"weekday_name": "Day of week", "complaint_count": "Complaints"},
)
fig.show()


In [ ]:
hour_weekday = q("""
SELECT
    strftime(complaint_from_date, '%w')::INTEGER AS weekday_num,
    strftime(complaint_from_date, '%A') AS weekday_name,
    EXTRACT(hour FROM complaint_from_ts)::INTEGER AS incident_hour,
    COUNT(*) AS complaint_count
FROM complaints_enriched
WHERE complaint_from_ts IS NOT NULL
GROUP BY 1, 2, 3
ORDER BY 1, 3
""")
hour_weekday_pivot = hour_weekday.pivot(index="weekday_name", columns="incident_hour", values="complaint_count")
weekday_order = ["Sunday", "Monday", "Tuesday", "Wednesday", "Thursday", "Friday", "Saturday"]
hour_weekday_pivot = hour_weekday_pivot.reindex(weekday_order)

fig = px.imshow(
    hour_weekday_pivot,
    aspect="auto",
    color_continuous_scale="Plasma",
    title="Complaint Volume by Day of Week and Hour",
    labels={"x": "Hour", "y": "Day of week", "color": "Complaints"},
)
fig.show()


In [ ]:
duration_lag_summary = q("""
WITH intervals AS (
    SELECT
        date_diff('hour', complaint_from_ts, complaint_to_ts) AS duration_hours,
        date_diff('day', complaint_from_date, report_date) AS report_lag_days
    FROM complaints_enriched
    WHERE complaint_from_date IS NOT NULL
)
SELECT
    COUNT(*) AS rows_with_from_date,
    SUM(CASE WHEN duration_hours IS NULL THEN 1 ELSE 0 END) AS missing_duration,
    SUM(CASE WHEN duration_hours < 0 THEN 1 ELSE 0 END) AS negative_duration_rows,
    SUM(CASE WHEN duration_hours > 24 * 30 THEN 1 ELSE 0 END) AS duration_over_30_days,
    quantile_cont(duration_hours, 0.50) AS median_duration_hours,
    quantile_cont(duration_hours, 0.90) AS p90_duration_hours,
    quantile_cont(duration_hours, 0.99) AS p99_duration_hours,
    SUM(CASE WHEN report_lag_days < 0 THEN 1 ELSE 0 END) AS negative_report_lag_rows,
    quantile_cont(report_lag_days, 0.50) AS median_report_lag_days,
    quantile_cont(report_lag_days, 0.90) AS p90_report_lag_days,
    quantile_cont(report_lag_days, 0.99) AS p99_report_lag_days
FROM intervals
""")
show_df(duration_lag_summary)


In [ ]:
lag_distribution = q("""
WITH lagged AS (
    SELECT date_diff('day', complaint_from_date, report_date) AS report_lag_days
    FROM complaints_enriched
    WHERE complaint_from_date IS NOT NULL AND report_date IS NOT NULL
)
SELECT
    CASE
        WHEN report_lag_days < 0 THEN 'negative'
        WHEN report_lag_days = 0 THEN 'same day'
        WHEN report_lag_days BETWEEN 1 AND 7 THEN '1-7 days'
        WHEN report_lag_days BETWEEN 8 AND 30 THEN '8-30 days'
        WHEN report_lag_days BETWEEN 31 AND 365 THEN '31-365 days'
        ELSE 'over 365 days'
    END AS report_lag_bucket,
    COUNT(*) AS complaint_count
FROM lagged
GROUP BY 1
ORDER BY complaint_count DESC
""")
show_df(lag_distribution)

fig = px.bar(
    lag_distribution,
    x="report_lag_bucket",
    y="complaint_count",
    title="Report Lag Buckets",
    labels={"report_lag_bucket": "Report lag bucket", "complaint_count": "Complaints"},
)
fig.show()


# 3. Offense, Law Category, and Complaint Classification

This section examines offense hierarchy, law category distribution, completed versus attempted complaints, and category consistency.


In [ ]:
law_counts = top_values("LAW_CAT_CD", 20)
show_df(law_counts)
fig = px.bar(law_counts, x="value", y="row_count", title="Law Category Distribution", labels={"value": "Law category", "row_count": "Complaints"})
fig.show()

attempt_counts = top_values("CRM_ATPT_CPTD_CD", 20)
show_df(attempt_counts)
fig = px.bar(attempt_counts, x="value", y="row_count", title="Completed vs Attempted Distribution", labels={"value": "Status", "row_count": "Complaints"})
fig.show()


In [ ]:
top_offenses = top_values("OFNS_DESC", TOP_N)
show_df(top_offenses, TOP_N)
fig = px.bar(
    top_offenses.sort_values("row_count"),
    x="row_count",
    y="value",
    orientation="h",
    title=f"Top {TOP_N} Offense Descriptions",
    labels={"value": "Offense", "row_count": "Complaints"},
    height=650,
)
fig.show()


In [ ]:
top_pd_desc = top_values("PD_DESC", TOP_N)
show_df(top_pd_desc, TOP_N)
fig = px.bar(
    top_pd_desc.sort_values("row_count"),
    x="row_count",
    y="value",
    orientation="h",
    title=f"Top {TOP_N} PD Descriptions",
    labels={"value": "PD description", "row_count": "Complaints"},
    height=650,
)
fig.show()


In [ ]:
offense_law = q("""
SELECT
    OFNS_DESC,
    LAW_CAT_CD,
    COUNT(*) AS complaint_count
FROM complaints_enriched
WHERE OFNS_DESC IS NOT NULL AND LAW_CAT_CD IS NOT NULL
GROUP BY 1, 2
ORDER BY complaint_count DESC
""")
show_df(offense_law, 30)

top_offense_names = top_offenses["value"].dropna().head(15).tolist()
offense_law_top = offense_law[offense_law["OFNS_DESC"].isin(top_offense_names)]
offense_law_pivot = offense_law_top.pivot_table(index="OFNS_DESC", columns="LAW_CAT_CD", values="complaint_count", aggfunc="sum", fill_value=0)
fig = px.imshow(
    offense_law_pivot,
    aspect="auto",
    color_continuous_scale="Blues",
    title="Top Offenses by Law Category",
    labels={"x": "Law category", "y": "Offense", "color": "Complaints"},
)
fig.show()


In [ ]:
classification_consistency = q("""
WITH ky_map AS (
    SELECT KY_CD, COUNT(DISTINCT OFNS_DESC) AS offense_desc_count, COUNT(*) AS row_count
    FROM complaints_enriched
    WHERE KY_CD IS NOT NULL
    GROUP BY 1
),
pd_map AS (
    SELECT PD_CD, COUNT(DISTINCT PD_DESC) AS pd_desc_count, COUNT(*) AS row_count
    FROM complaints_enriched
    WHERE PD_CD IS NOT NULL
    GROUP BY 1
)
SELECT 'KY_CD_to_OFNS_DESC' AS mapping_check, COUNT(*) AS code_count, SUM(CASE WHEN offense_desc_count > 1 THEN 1 ELSE 0 END) AS codes_with_multiple_descriptions
FROM ky_map
UNION ALL
SELECT 'PD_CD_to_PD_DESC' AS mapping_check, COUNT(*) AS code_count, SUM(CASE WHEN pd_desc_count > 1 THEN 1 ELSE 0 END) AS codes_with_multiple_descriptions
FROM pd_map
""")
show_df(classification_consistency)

ky_inconsistencies = q("""
SELECT KY_CD, COUNT(DISTINCT OFNS_DESC) AS offense_desc_count, COUNT(*) AS row_count,
       string_agg(DISTINCT OFNS_DESC, ' | ') AS offense_descriptions
FROM complaints_enriched
WHERE KY_CD IS NOT NULL
GROUP BY 1
HAVING COUNT(DISTINCT OFNS_DESC) > 1
ORDER BY row_count DESC
LIMIT 30
""")
show_df(ky_inconsistencies, 30)


In [ ]:
invalid_law_categories = q("""
SELECT LAW_CAT_CD, COUNT(*) AS complaint_count
FROM complaints_enriched
WHERE LAW_CAT_CD IS NOT NULL
  AND LAW_CAT_CD NOT IN ('FELONY', 'MISDEMEANOR', 'VIOLATION')
GROUP BY 1
ORDER BY complaint_count DESC
""")
show_df(invalid_law_categories)


# 4. Geography, Boroughs, Precincts, Coordinates, and Hotspots

This section audits administrative geography and coordinate quality. Valid coordinates are checked against a broad NYC bounding box. This is not a substitute for polygon-level borough validation, but it is a strong first-pass quality screen.


In [ ]:
borough_counts = top_values("BORO_NM", 20)
show_df(borough_counts)
fig = px.bar(borough_counts, x="value", y="row_count", title="Borough Distribution", labels={"value": "Borough", "row_count": "Complaints"})
fig.show()

precinct_counts = top_values("ADDR_PCT_CD", TOP_N)
show_df(precinct_counts, TOP_N)
fig = px.bar(
    precinct_counts.sort_values("row_count"),
    x="row_count",
    y="value",
    orientation="h",
    title=f"Top {TOP_N} Precincts by Complaint Volume",
    labels={"value": "Precinct", "row_count": "Complaints"},
    height=650,
)
fig.show()


In [ ]:
geo_quality = q(f"""
SELECT
    COUNT(*) AS row_count,
    SUM(CASE WHEN latitude_num IS NULL OR longitude_num IS NULL THEN 1 ELSE 0 END) AS missing_lat_or_lon,
    SUM(CASE WHEN latitude_num = 0 OR longitude_num = 0 THEN 1 ELSE 0 END) AS zero_lat_or_lon,
    SUM(CASE WHEN has_valid_nyc_coordinates THEN 1 ELSE 0 END) AS valid_nyc_coordinates,
    SUM(CASE WHEN latitude_num IS NOT NULL AND longitude_num IS NOT NULL AND NOT has_valid_nyc_coordinates THEN 1 ELSE 0 END) AS coordinates_outside_nyc_bounds,
    MIN(latitude_num) AS min_latitude,
    MAX(latitude_num) AS max_latitude,
    MIN(longitude_num) AS min_longitude,
    MAX(longitude_num) AS max_longitude,
    SUM(CASE WHEN X_COORD_CD IS NULL OR Y_COORD_CD IS NULL THEN 1 ELSE 0 END) AS missing_projected_coordinates
FROM complaints_enriched
""")
GEO_QUALITY_SAFE_COLUMNS = [
    "row_count",
    "missing_lat_or_lon",
    "zero_lat_or_lon",
    "valid_nyc_coordinates",
    "coordinates_outside_nyc_bounds",
    "missing_projected_coordinates",
]
geo_quality_safe = geo_quality[GEO_QUALITY_SAFE_COLUMNS].copy()
show_df(geo_quality_safe)


In [ ]:
borough_precinct = q("""
SELECT
    BORO_NM,
    ADDR_PCT_CD,
    COUNT(*) AS complaint_count
FROM complaints_enriched
WHERE BORO_NM IS NOT NULL AND ADDR_PCT_CD IS NOT NULL
GROUP BY 1, 2
ORDER BY complaint_count DESC
""")
show_df(borough_precinct, 40)

borough_precinct_top = borough_precinct.groupby("BORO_NM", group_keys=False).head(20)
fig = px.bar(
    borough_precinct_top,
    x="ADDR_PCT_CD",
    y="complaint_count",
    color="BORO_NM",
    title="Top Precincts Within Each Borough",
    labels={"ADDR_PCT_CD": "Precinct", "complaint_count": "Complaints", "BORO_NM": "Borough"},
    height=550,
)
fig.show()


In [ ]:
geo_coverage_by_category = q("""
SELECT
    COALESCE(BORO_NM, 'UNKNOWN') AS borough,
    COALESCE(LAW_CAT_CD, 'UNKNOWN') AS law_category,
    COALESCE(OFNS_DESC, 'UNKNOWN') AS offense_type,
    COUNT(*) AS complaint_count
FROM complaints_enriched
WHERE has_valid_nyc_coordinates
GROUP BY 1, 2, 3
HAVING COUNT(*) >= 10
ORDER BY complaint_count DESC
LIMIT 50
""")
show_df(geo_coverage_by_category, 50)


In [ ]:
# Coordinate density grid. Rounding to 3 decimals is roughly a neighborhood/block-level first pass.
coord_grid = q("""
SELECT
    ROUND(latitude_num, 3) AS lat_grid,
    ROUND(longitude_num, 3) AS lon_grid,
    COUNT(*) AS complaint_count,
    COUNT(DISTINCT OFNS_DESC) AS offense_type_count,
    COUNT(DISTINCT BORO_NM) AS borough_count
FROM complaints_enriched
WHERE has_valid_nyc_coordinates
GROUP BY 1, 2
HAVING COUNT(*) >= 10
ORDER BY complaint_count DESC
LIMIT 100
""")
show_df(coord_grid, 30)

fig = px.scatter_mapbox(
    coord_grid,
    lat="lat_grid",
    lon="lon_grid",
    size="complaint_count",
    color="complaint_count",
    color_continuous_scale="Inferno",
    zoom=9,
    height=650,
    title="Top Coordinate Density Cells",
)
fig.update_layout(mapbox_style="open-street-map", margin={"r": 0, "t": 45, "l": 0, "b": 0})
fig.show()


### Coordinate Privacy Note

Event-level coordinate previews and heatmaps are intentionally omitted. The rounded, minimum-count coordinate grid above is the notebook's public geographic example.


# 5. Premise, Occurrence Context, Jurisdiction, Transit, Parks, and Housing

These fields help the dashboard explain where and under what operational context complaints occur. Some are sparse by design.


In [ ]:
context_columns = ["PREM_TYP_DESC", "LOC_OF_OCCUR_DESC", "JURIS_DESC", "PATROL_BORO", "PARKS_NM", "HADEVELOPT", "HOUSING_PSA", "TRANSIT_DISTRICT", "STATION_NAME"]
for column in [col for col in context_columns if col in raw_columns]:
    display(Markdown(f"### Context distribution: `{column}`"))
    tv = top_values(column, TOP_N)
    show_df(tv, TOP_N)
    fig = px.bar(
        tv.sort_values("row_count"),
        x="row_count",
        y="value",
        orientation="h",
        title=f"Top {TOP_N}: {column}",
        labels={"row_count": "Complaints", "value": column},
        height=600,
    )
    fig.show()


In [ ]:
premise_by_law = q("""
SELECT
    PREM_TYP_DESC,
    LAW_CAT_CD,
    COUNT(*) AS complaint_count
FROM complaints_enriched
WHERE PREM_TYP_DESC IS NOT NULL AND LAW_CAT_CD IS NOT NULL
GROUP BY 1, 2
ORDER BY complaint_count DESC
LIMIT 300
""")

top_premises = top_values("PREM_TYP_DESC", 15)["value"].dropna().tolist()
premise_by_law_top = premise_by_law[premise_by_law["PREM_TYP_DESC"].isin(top_premises)]
premise_law_pivot = premise_by_law_top.pivot_table(index="PREM_TYP_DESC", columns="LAW_CAT_CD", values="complaint_count", aggfunc="sum", fill_value=0)
fig = px.imshow(
    premise_law_pivot,
    aspect="auto",
    color_continuous_scale="Greens",
    title="Top Premise Types by Law Category",
    labels={"x": "Law category", "y": "Premise type", "color": "Complaints"},
)
fig.show()


# 6. Victim and Suspect Field Audit

These fields are sensitive. They are audited for completeness, unknown values, invalid values, and coverage differences. They should not be used as input features for the first forecasting model.


In [ ]:
SENSITIVE_COLUMNS = ["VIC_AGE_GROUP", "VIC_RACE", "VIC_SEX", "SUSP_AGE_GROUP", "SUSP_RACE", "SUSP_SEX"]
sensitive_profile = column_profile_public[column_profile_public["column_name"].isin(SENSITIVE_COLUMNS)].copy()
show_df(sensitive_profile, 20)

fig = px.bar(
    sensitive_profile.sort_values("missing_pct"),
    x="missing_pct",
    y="column_name",
    orientation="h",
    title="Missingness in Sensitive Demographic Fields",
    labels={"missing_pct": "Missing (%)", "column_name": "Field"},
    height=400,
)
fig.show()


In [ ]:
for column in [col for col in SENSITIVE_COLUMNS if col in raw_columns]:
    display(Markdown(f"### Sensitive field distribution: `{column}`"))
    tv = top_values(column, TOP_N)
    show_df(tv, TOP_N)
    fig = px.bar(
        tv.sort_values("row_count"),
        x="row_count",
        y="value",
        orientation="h",
        title=f"Distribution: {column}",
        labels={"row_count": "Complaints", "value": column},
        height=550,
    )
    fig.show()


In [ ]:
valid_age_groups = {"<18", "18-24", "25-44", "45-64", "65+", "UNKNOWN"}
valid_sex_values = {"F", "M", "U", "UNKNOWN", "D", "E"}

age_quality = q("""
SELECT
    'VIC_AGE_GROUP' AS field_name,
    COUNT(*) AS rows_checked,
    SUM(CASE WHEN VIC_AGE_GROUP IS NULL THEN 1 ELSE 0 END) AS missing_rows,
    SUM(CASE WHEN VIC_AGE_GROUP IS NOT NULL AND VIC_AGE_GROUP NOT IN ('<18', '18-24', '25-44', '45-64', '65+', 'UNKNOWN') THEN 1 ELSE 0 END) AS unexpected_value_rows
FROM complaints_enriched
UNION ALL
SELECT
    'SUSP_AGE_GROUP' AS field_name,
    COUNT(*) AS rows_checked,
    SUM(CASE WHEN SUSP_AGE_GROUP IS NULL THEN 1 ELSE 0 END) AS missing_rows,
    SUM(CASE WHEN SUSP_AGE_GROUP IS NOT NULL AND SUSP_AGE_GROUP NOT IN ('<18', '18-24', '25-44', '45-64', '65+', 'UNKNOWN') THEN 1 ELSE 0 END) AS unexpected_value_rows
FROM complaints_enriched
UNION ALL
SELECT
    'VIC_SEX' AS field_name,
    COUNT(*) AS rows_checked,
    SUM(CASE WHEN VIC_SEX IS NULL THEN 1 ELSE 0 END) AS missing_rows,
    SUM(CASE WHEN VIC_SEX IS NOT NULL AND VIC_SEX NOT IN ('F', 'M', 'U', 'UNKNOWN', 'D', 'E') THEN 1 ELSE 0 END) AS unexpected_value_rows
FROM complaints_enriched
UNION ALL
SELECT
    'SUSP_SEX' AS field_name,
    COUNT(*) AS rows_checked,
    SUM(CASE WHEN SUSP_SEX IS NULL THEN 1 ELSE 0 END) AS missing_rows,
    SUM(CASE WHEN SUSP_SEX IS NOT NULL AND SUSP_SEX NOT IN ('F', 'M', 'U', 'UNKNOWN', 'D', 'E') THEN 1 ELSE 0 END) AS unexpected_value_rows
FROM complaints_enriched
""")
show_df(age_quality)


# 7. Cross-Sectional Patterns

This section combines time, geography, and offense categories to identify the slices that should be prioritized for the dashboard and later forecasting tables.


In [ ]:
borough_law = q("""
SELECT
    BORO_NM,
    LAW_CAT_CD,
    COUNT(*) AS complaint_count
FROM complaints_enriched
WHERE BORO_NM IS NOT NULL AND LAW_CAT_CD IS NOT NULL
GROUP BY 1, 2
ORDER BY complaint_count DESC
""")
show_df(borough_law, 30)

borough_law_pivot = borough_law.pivot_table(index="BORO_NM", columns="LAW_CAT_CD", values="complaint_count", aggfunc="sum", fill_value=0)
fig = px.imshow(
    borough_law_pivot,
    aspect="auto",
    color_continuous_scale="OrRd",
    title="Borough by Law Category",
    labels={"x": "Law category", "y": "Borough", "color": "Complaints"},
)
fig.show()


In [ ]:
borough_offense = q("""
SELECT
    BORO_NM,
    OFNS_DESC,
    COUNT(*) AS complaint_count
FROM complaints_enriched
WHERE BORO_NM IS NOT NULL AND OFNS_DESC IS NOT NULL
GROUP BY 1, 2
ORDER BY complaint_count DESC
""")

# Show top offenses within each borough.
borough_offense_ranked = borough_offense.copy()
borough_offense_ranked["rank_within_borough"] = borough_offense_ranked.groupby("BORO_NM")["complaint_count"].rank(method="first", ascending=False)
show_df(borough_offense_ranked[borough_offense_ranked["rank_within_borough"] <= 10], 80)

borough_offense_top = borough_offense_ranked[borough_offense_ranked["rank_within_borough"] <= 8]
fig = px.bar(
    borough_offense_top,
    x="complaint_count",
    y="OFNS_DESC",
    color="BORO_NM",
    facet_col="BORO_NM",
    facet_col_wrap=2,
    orientation="h",
    title="Top Offenses Within Each Borough",
    height=900,
)
fig.update_yaxes(matches=None)
fig.show()


In [ ]:
offense_hour = q("""
SELECT
    OFNS_DESC,
    EXTRACT(hour FROM complaint_from_ts)::INTEGER AS incident_hour,
    COUNT(*) AS complaint_count
FROM complaints_enriched
WHERE complaint_from_ts IS NOT NULL AND OFNS_DESC IS NOT NULL
GROUP BY 1, 2
""")

top_offense_names_12 = top_offenses["value"].dropna().head(12).tolist()
offense_hour_top = offense_hour[offense_hour["OFNS_DESC"].isin(top_offense_names_12)]
offense_hour_pivot = offense_hour_top.pivot_table(index="OFNS_DESC", columns="incident_hour", values="complaint_count", aggfunc="sum", fill_value=0)
fig = px.imshow(
    offense_hour_pivot,
    aspect="auto",
    color_continuous_scale="Magma",
    title="Top Offenses by Incident Hour",
    labels={"x": "Hour", "y": "Offense", "color": "Complaints"},
)
fig.show()


In [ ]:
monthly_top_offenses = q("""
SELECT
    date_trunc('month', complaint_from_date)::DATE AS month_start,
    OFNS_DESC,
    COUNT(*) AS complaint_count
FROM complaints_enriched
WHERE complaint_from_date IS NOT NULL AND OFNS_DESC IS NOT NULL
GROUP BY 1, 2
ORDER BY 1, 3 DESC
""")
monthly_top_offenses = monthly_top_offenses[monthly_top_offenses["OFNS_DESC"].isin(top_offense_names_12)]

fig = px.line(
    monthly_top_offenses,
    x="month_start",
    y="complaint_count",
    color="OFNS_DESC",
    title="Monthly Trends for Top Offenses",
    labels={"month_start": "Month", "complaint_count": "Complaints", "OFNS_DESC": "Offense"},
    height=650,
)
fig.show()


# 8. Data Quality Rules and Issue Inventory

This section produces explicit issue counts for the rules most relevant to cleaning, modeling, and dashboard reliability.


In [ ]:
data_quality_rules = q(f"""
SELECT 'missing_complaint_number' AS rule_name, SUM(CASE WHEN CMPLNT_NUM IS NULL THEN 1 ELSE 0 END) AS issue_count FROM complaints_enriched
UNION ALL
SELECT 'duplicate_complaint_number_surplus' AS rule_name, COUNT(*) - COUNT(DISTINCT CMPLNT_NUM) AS issue_count FROM complaints_enriched WHERE CMPLNT_NUM IS NOT NULL
UNION ALL
SELECT 'missing_or_invalid_from_date' AS rule_name, SUM(CASE WHEN complaint_from_date IS NULL THEN 1 ELSE 0 END) AS issue_count FROM complaints_enriched
UNION ALL
SELECT 'missing_or_invalid_report_date' AS rule_name, SUM(CASE WHEN report_date IS NULL THEN 1 ELSE 0 END) AS issue_count FROM complaints_enriched
UNION ALL
SELECT 'report_date_before_incident_date' AS rule_name, SUM(CASE WHEN report_date < complaint_from_date THEN 1 ELSE 0 END) AS issue_count FROM complaints_enriched
UNION ALL
SELECT 'to_date_before_from_date' AS rule_name, SUM(CASE WHEN complaint_to_date < complaint_from_date THEN 1 ELSE 0 END) AS issue_count FROM complaints_enriched
UNION ALL
SELECT 'invalid_from_time' AS rule_name, SUM(CASE WHEN has_valid_from_time = false THEN 1 ELSE 0 END) AS issue_count FROM complaints_enriched
UNION ALL
SELECT 'missing_borough' AS rule_name, SUM(CASE WHEN BORO_NM IS NULL THEN 1 ELSE 0 END) AS issue_count FROM complaints_enriched
UNION ALL
SELECT 'missing_precinct' AS rule_name, SUM(CASE WHEN ADDR_PCT_CD IS NULL THEN 1 ELSE 0 END) AS issue_count FROM complaints_enriched
UNION ALL
SELECT 'missing_latitude_or_longitude' AS rule_name, SUM(CASE WHEN latitude_num IS NULL OR longitude_num IS NULL THEN 1 ELSE 0 END) AS issue_count FROM complaints_enriched
UNION ALL
SELECT 'coordinates_outside_nyc_bounds' AS rule_name, SUM(CASE WHEN latitude_num IS NOT NULL AND longitude_num IS NOT NULL AND NOT has_valid_nyc_coordinates THEN 1 ELSE 0 END) AS issue_count FROM complaints_enriched
UNION ALL
SELECT 'missing_offense_description' AS rule_name, SUM(CASE WHEN OFNS_DESC IS NULL THEN 1 ELSE 0 END) AS issue_count FROM complaints_enriched
UNION ALL
SELECT 'invalid_law_category' AS rule_name, SUM(CASE WHEN LAW_CAT_CD IS NOT NULL AND LAW_CAT_CD NOT IN ('FELONY', 'MISDEMEANOR', 'VIOLATION') THEN 1 ELSE 0 END) AS issue_count FROM complaints_enriched
""")
if not pd.isna(row_count):
    data_quality_rules["issue_pct"] = data_quality_rules["issue_count"] / row_count * 100
show_df(data_quality_rules.sort_values("issue_count", ascending=False), 50)

fig = px.bar(
    data_quality_rules.sort_values("issue_count", ascending=True),
    x="issue_count",
    y="rule_name",
    orientation="h",
    title="Data Quality Issue Counts",
    labels={"issue_count": "Rows affected", "rule_name": "Rule"},
    height=650,
)
fig.show()


In [ ]:
duplicate_complaint_number_summary = q("""
WITH duplicate_counts AS (
    SELECT CMPLNT_NUM, COUNT(*) AS duplicate_count
    FROM complaints_enriched
    WHERE CMPLNT_NUM IS NOT NULL
    GROUP BY 1
    HAVING COUNT(*) > 1
)
SELECT
    duplicate_count,
    COUNT(*) AS complaint_number_count
FROM duplicate_counts
GROUP BY 1
ORDER BY duplicate_count
""")
show_df(duplicate_complaint_number_summary, 50)


In [ ]:
invalid_coordinate_summary = q("""
SELECT
    CASE
        WHEN latitude_num IS NULL OR longitude_num IS NULL THEN 'missing'
        WHEN latitude_num = 0 OR longitude_num = 0 THEN 'zero'
        WHEN NOT has_valid_nyc_coordinates THEN 'outside_broad_nyc_bounds'
        ELSE 'valid'
    END AS coordinate_quality,
    COALESCE(BORO_NM, 'UNKNOWN') AS borough,
    COUNT(*) AS complaint_count
FROM complaints_enriched
WHERE latitude_num IS NULL
   OR longitude_num IS NULL
   OR latitude_num = 0
   OR longitude_num = 0
   OR NOT has_valid_nyc_coordinates
GROUP BY 1, 2
ORDER BY complaint_count DESC
""")
show_df(invalid_coordinate_summary, 50)


In [ ]:
invalid_date_summary = q("""
SELECT
    CASE
        WHEN complaint_from_date IS NULL THEN 'missing_or_invalid_from_date'
        WHEN report_date IS NULL THEN 'missing_or_invalid_report_date'
        WHEN report_date < complaint_from_date THEN 'report_date_before_incident_date'
        WHEN complaint_to_date < complaint_from_date THEN 'to_date_before_from_date'
    END AS date_quality_issue,
    COALESCE(BORO_NM, 'UNKNOWN') AS borough,
    COUNT(*) AS complaint_count
FROM complaints_enriched
WHERE complaint_from_date IS NULL
   OR report_date IS NULL
   OR report_date < complaint_from_date
   OR complaint_to_date < complaint_from_date
GROUP BY 1, 2
ORDER BY complaint_count DESC
""")
show_df(invalid_date_summary, 50)


# 9. Statistical Association Checks on the Sample

These sample-based checks are not used for causal claims. They are meant to identify strong categorical relationships, redundancy, and potential leakage or proxy variables.


In [ ]:
from scipy.stats import chi2_contingency


def cramers_v(x: pd.Series, y: pd.Series) -> float:
    confusion = pd.crosstab(x.fillna("__MISSING__"), y.fillna("__MISSING__"))
    if confusion.empty or min(confusion.shape) < 2:
        return np.nan
    chi2 = chi2_contingency(confusion, correction=False)[0]
    n = confusion.to_numpy().sum()
    r, k = confusion.shape
    phi2 = chi2 / n
    phi2_corr = max(0, phi2 - ((k - 1) * (r - 1)) / max(n - 1, 1))
    r_corr = r - ((r - 1) ** 2) / max(n - 1, 1)
    k_corr = k - ((k - 1) ** 2) / max(n - 1, 1)
    denom = min(k_corr - 1, r_corr - 1)
    return np.sqrt(phi2_corr / denom) if denom > 0 else np.nan

association_columns = [
    "BORO_NM", "ADDR_PCT_CD", "PATROL_BORO", "OFNS_DESC", "LAW_CAT_CD", "CRM_ATPT_CPTD_CD",
    "PREM_TYP_DESC", "LOC_OF_OCCUR_DESC", "JURIS_DESC", "VIC_AGE_GROUP", "VIC_SEX", "SUSP_AGE_GROUP", "SUSP_SEX",
]
association_columns = [column for column in association_columns if column in sample_df.columns]
association_sample = sample_df[association_columns].copy()
if len(association_sample) > 60_000:
    association_sample = association_sample.sample(60_000, random_state=RANDOM_SEED)

assoc_records = []
for i, left in enumerate(association_columns):
    for right in association_columns[i + 1:]:
        assoc_records.append({"left": left, "right": right, "cramers_v": cramers_v(association_sample[left], association_sample[right])})

associations = pd.DataFrame(assoc_records).sort_values("cramers_v", ascending=False)
show_df(associations, 40)


In [ ]:
assoc_matrix = pd.DataFrame(index=association_columns, columns=association_columns, dtype=float)
for column in association_columns:
    assoc_matrix.loc[column, column] = 1.0
for _, row in associations.iterrows():
    assoc_matrix.loc[row["left"], row["right"]] = row["cramers_v"]
    assoc_matrix.loc[row["right"], row["left"]] = row["cramers_v"]

fig = px.imshow(
    assoc_matrix.astype(float),
    zmin=0,
    zmax=1,
    color_continuous_scale="Cividis",
    title="Sample-Based Cramer's V Between Categorical Fields",
    labels={"color": "Cramer's V"},
    height=700,
)
fig.show()


# 10. Trend Change and Early Anomaly Exploration

This is not a final anomaly model. It identifies recent changes, rolling deviations, and borough/offense combinations that deserve dashboard attention.


In [ ]:
latest_dates = q("""
SELECT
    MAX(complaint_from_date) AS max_incident_date,
    MAX(report_date) AS max_report_date
FROM complaints_enriched
WHERE complaint_from_date IS NOT NULL
""")
show_df(latest_dates)


In [ ]:
recent_vs_prior = q("""
WITH bounds AS (
    SELECT MAX(complaint_from_date) AS max_date
    FROM complaints_enriched
    WHERE complaint_from_date IS NOT NULL
), labeled AS (
    SELECT
        BORO_NM,
        OFNS_DESC,
        CASE
            WHEN complaint_from_date > max_date - INTERVAL 90 DAY THEN 'recent_90_days'
            WHEN complaint_from_date > max_date - INTERVAL 180 DAY THEN 'prior_90_days'
            ELSE NULL
        END AS period_label
    FROM complaints_enriched, bounds
    WHERE complaint_from_date IS NOT NULL
      AND BORO_NM IS NOT NULL
      AND OFNS_DESC IS NOT NULL
), aggregated AS (
    SELECT BORO_NM, OFNS_DESC, period_label, COUNT(*) AS complaint_count
    FROM labeled
    WHERE period_label IS NOT NULL
    GROUP BY 1, 2, 3
), pivoted AS (
    SELECT
        BORO_NM,
        OFNS_DESC,
        SUM(CASE WHEN period_label = 'recent_90_days' THEN complaint_count ELSE 0 END) AS recent_90_days,
        SUM(CASE WHEN period_label = 'prior_90_days' THEN complaint_count ELSE 0 END) AS prior_90_days
    FROM aggregated
    GROUP BY 1, 2
)
SELECT
    *,
    recent_90_days - prior_90_days AS absolute_change,
    100.0 * (recent_90_days - prior_90_days) / NULLIF(prior_90_days, 0) AS pct_change
FROM pivoted
WHERE recent_90_days + prior_90_days >= 100
ORDER BY absolute_change DESC
LIMIT 100
""")

recent_vs_prior["change_magnitude"] = recent_vs_prior["absolute_change"].abs()
recent_vs_prior["change_direction"] = np.select(
    [recent_vs_prior["absolute_change"] > 0, recent_vs_prior["absolute_change"] < 0],
    ["increase", "decrease"],
    default="no change",
)

show_df(recent_vs_prior, 50)

fig = px.scatter(
    recent_vs_prior.head(60),
    x="prior_90_days",
    y="recent_90_days",
    size="change_magnitude",
    color="BORO_NM",
    symbol="change_direction",
    hover_data={
        "OFNS_DESC": True,
        "pct_change": ":.2f",
        "absolute_change": True,
        "change_magnitude": False,
    },
    title="Recent 90 Days vs Previous 90 Days by Borough and Offense",
    labels={
        "prior_90_days": "Previous 90 days",
        "recent_90_days": "Recent 90 days",
        "change_magnitude": "Change magnitude",
    },
    height=600,
)

fig.add_trace(
    go.Scatter(
        x=[0, recent_vs_prior[["prior_90_days", "recent_90_days"]].max().max()],
        y=[0, recent_vs_prior[["prior_90_days", "recent_90_days"]].max().max()],
        mode="lines",
        name="No change",
    )
)

fig.show()

In [ ]:
rolling_anomalies = q("""
WITH monthly AS (
    SELECT
        date_trunc('month', complaint_from_date)::DATE AS month_start,
        BORO_NM,
        OFNS_DESC,
        COUNT(*) AS complaint_count
    FROM complaints_enriched
    WHERE complaint_from_date IS NOT NULL
      AND BORO_NM IS NOT NULL
      AND OFNS_DESC IS NOT NULL
    GROUP BY 1, 2, 3
), scored AS (
    SELECT
        *,
        AVG(complaint_count) OVER (
            PARTITION BY BORO_NM, OFNS_DESC
            ORDER BY month_start
            ROWS BETWEEN 12 PRECEDING AND 1 PRECEDING
        ) AS trailing_12_month_avg,
        STDDEV_SAMP(complaint_count) OVER (
            PARTITION BY BORO_NM, OFNS_DESC
            ORDER BY month_start
            ROWS BETWEEN 12 PRECEDING AND 1 PRECEDING
        ) AS trailing_12_month_std,
        COUNT(*) OVER (
            PARTITION BY BORO_NM, OFNS_DESC
            ORDER BY month_start
            ROWS BETWEEN 12 PRECEDING AND 1 PRECEDING
        ) AS trailing_month_count
    FROM monthly
)
SELECT
    month_start,
    BORO_NM,
    OFNS_DESC,
    complaint_count,
    trailing_12_month_avg,
    trailing_12_month_std,
    (complaint_count - trailing_12_month_avg) / NULLIF(trailing_12_month_std, 0) AS z_score
FROM scored
WHERE trailing_month_count >= 6
  AND trailing_12_month_std > 0
ORDER BY z_score DESC
LIMIT 100
""")
show_df(rolling_anomalies, 50)

fig = px.bar(
    rolling_anomalies.head(30).sort_values("z_score"),
    x="z_score",
    y="OFNS_DESC",
    color="BORO_NM",
    orientation="h",
    title="Highest Positive Monthly Deviations vs Trailing 12-Month Pattern",
    labels={"z_score": "Z-score", "OFNS_DESC": "Offense"},
    height=800,
)
fig.show()


# 11. Modeling and Dashboard Readiness

This section creates aggregate tables that match the roadmap's first modeling target: weekly and monthly complaint counts by area and offense type. These outputs are intended for baseline forecasting, dashboard charts, and anomaly layers.


In [ ]:
weekly_output_path = PROCESSED_DIR / "crime_weekly_area.parquet"
monthly_output_path = PROCESSED_DIR / "crime_monthly_area.parquet"

con.execute(f"""
COPY (
    SELECT
        date_trunc('week', complaint_from_date)::DATE AS week_start,
        COALESCE(BORO_NM, 'UNKNOWN') AS borough,
        COALESCE(ADDR_PCT_CD, 'UNKNOWN') AS precinct,
        COALESCE(OFNS_DESC, 'UNKNOWN') AS offense_type,
        COALESCE(LAW_CAT_CD, 'UNKNOWN') AS law_category,
        COUNT(*) AS crime_count
    FROM complaints_enriched
    WHERE complaint_from_date IS NOT NULL
    GROUP BY 1, 2, 3, 4, 5
) TO {sql_string(weekly_output_path)} (FORMAT PARQUET)
""")

con.execute(f"""
COPY (
    SELECT
        date_trunc('month', complaint_from_date)::DATE AS month_start,
        COALESCE(BORO_NM, 'UNKNOWN') AS borough,
        COALESCE(ADDR_PCT_CD, 'UNKNOWN') AS precinct,
        COALESCE(OFNS_DESC, 'UNKNOWN') AS offense_type,
        COALESCE(LAW_CAT_CD, 'UNKNOWN') AS law_category,
        COUNT(*) AS crime_count
    FROM complaints_enriched
    WHERE complaint_from_date IS NOT NULL
    GROUP BY 1, 2, 3, 4, 5
) TO {sql_string(monthly_output_path)} (FORMAT PARQUET)
""")

print(f"Wrote weekly aggregate: {weekly_output_path.relative_to(PROJECT_ROOT).as_posix()}")
print(f"Wrote monthly aggregate: {monthly_output_path.relative_to(PROJECT_ROOT).as_posix()}")


In [ ]:
weekly_area_preview = pd.read_parquet(weekly_output_path)
monthly_area_preview = pd.read_parquet(monthly_output_path)
print("Weekly aggregate shape:", weekly_area_preview.shape)
print("Monthly aggregate shape:", monthly_area_preview.shape)
show_df(weekly_area_preview.sort_values("crime_count", ascending=False), 20)
show_df(monthly_area_preview.sort_values("crime_count", ascending=False), 20)


In [ ]:
model_feature_recommendations = pd.DataFrame([
    {"field_group": "time", "recommended_fields": "week_start, month, year, week_of_year, season", "status": "use"},
    {"field_group": "location", "recommended_fields": "borough, precinct, patrol_boro, valid coordinate grid for hotspot layers", "status": "use with quality flags"},
    {"field_group": "offense", "recommended_fields": "offense_type, law_category, completed_attempted", "status": "use"},
    {"field_group": "history", "recommended_fields": "lag counts, rolling means, rolling std, prior-year same-week count", "status": "derive from aggregates"},
    {"field_group": "demographics", "recommended_fields": "victim and suspect age/race/sex", "status": "exclude from first forecasting model"},
    {"field_group": "raw identifiers", "recommended_fields": "complaint number", "status": "exclude from model; use for deduplication only"},
])
show_df(model_feature_recommendations, 20)


# 12. Export Schema Profile, Dashboard Summary, and Data Quality Report

The following cells write durable artifacts for the project repository. They summarize what was found and provide inputs for the next cleaning and modeling phases.


In [ ]:
def table_to_markdown(df: pd.DataFrame, max_rows: int = 30) -> str:
    if df is None or df.empty:
        return "No rows."
    shown = df.head(max_rows).copy()
    try:
        return shown.to_markdown(index=False)
    except Exception:
        return shown.to_string(index=False)

schema_profile_payload = {
    "generated_at_utc": datetime.now(timezone.utc).isoformat(),
    "source_file": RAW_CSV_PATH.relative_to(PROJECT_ROOT).as_posix(),
    "file_size_gb": file_size_gb,
    "row_count": None if pd.isna(row_count) else int(row_count),
    "column_count": len(raw_columns),
    "expected_missing_columns": missing_expected,
    "unexpected_columns": unexpected_columns,
    "column_groups": EXPECTED_COLUMN_GROUPS,
    "columns": column_profile_schema.replace({np.nan: None}).to_dict(orient="records"),
    "data_quality_rules": data_quality_rules.replace({np.nan: None}).to_dict(orient="records"),
}

schema_profile_path = PROCESSED_DIR / "schema_profile.json"
with open(schema_profile_path, "w", encoding="utf-8") as file:
    json.dump(schema_profile_payload, file, indent=2, ensure_ascii=False, default=str)

print(f"Wrote schema profile: {schema_profile_path.relative_to(PROJECT_ROOT).as_posix()}")


In [ ]:
dashboard_summary = {
    "generated_at_utc": datetime.now(timezone.utc).isoformat(),
    "source_file": RAW_CSV_PATH.relative_to(PROJECT_ROOT).as_posix(),
    "row_count": None if pd.isna(row_count) else int(row_count),
    "date_quality": date_quality.replace({np.nan: None}).to_dict(orient="records"),
    "geo_quality": geo_quality_safe.replace({np.nan: None}).to_dict(orient="records"),
    "top_boroughs": borough_counts.replace({np.nan: None}).to_dict(orient="records"),
    "top_offenses": top_offenses.replace({np.nan: None}).to_dict(orient="records"),
    "law_categories": law_counts.replace({np.nan: None}).to_dict(orient="records"),
    "weekly_aggregate_path": weekly_output_path.relative_to(PROJECT_ROOT).as_posix(),
    "monthly_aggregate_path": monthly_output_path.relative_to(PROJECT_ROOT).as_posix(),
}

dashboard_summary_path = PROCESSED_DIR / "dashboard_summary.json"
with open(dashboard_summary_path, "w", encoding="utf-8") as file:
    json.dump(dashboard_summary, file, indent=2, ensure_ascii=False, default=str)

print(f"Wrote dashboard summary: {dashboard_summary_path.relative_to(PROJECT_ROOT).as_posix()}")


In [ ]:
report_sections = []
report_sections.append("# NYPD Complaint Data Historic - Data Quality Report\n")
report_sections.append(f"Generated at UTC: `{datetime.now(timezone.utc).isoformat()}`\n")
report_sections.append("## Source\n")
report_sections.append(f"- File: `{RAW_CSV_PATH.relative_to(PROJECT_ROOT).as_posix()}`\n")
report_sections.append(f"- File size GB: `{file_size_gb:,.2f}`\n")
report_sections.append(f"- Row count: `{int(row_count):,}`\n" if not pd.isna(row_count) else "- Row count: skipped\n")
report_sections.append(f"- Column count: `{len(raw_columns)}`\n")

report_sections.append("\n## Column Profile\n")
report_sections.append(
    "Lexical minima and maxima are omitted because they can reproduce event identifiers, "
    "exact coordinates, or location examples.\n"
)
report_sections.append(table_to_markdown(column_profile_public, 80))

report_sections.append("\n\n## Data Quality Rule Counts\n")
report_sections.append(table_to_markdown(data_quality_rules.sort_values("issue_count", ascending=False), 80))

report_sections.append("\n\n## Date Quality\n")
report_sections.append(table_to_markdown(date_quality, 20))

report_sections.append("\n\n## Time Quality\n")
report_sections.append(table_to_markdown(time_quality, 20))

report_sections.append("\n\n## Geographic Quality\n")
report_sections.append(
    "Coordinate extrema are intentionally omitted because they can reproduce event-derived locations. "
    "Aggregate quality counts are retained.\n"
)
report_sections.append(table_to_markdown(geo_quality_safe, 20))

report_sections.append("\n\n## Missingness by Business Column Group\n")
report_sections.append(table_to_markdown(group_profile, 30))

report_sections.append("\n\n## Top Boroughs\n")
report_sections.append(table_to_markdown(borough_counts, 20))

report_sections.append("\n\n## Top Offenses\n")
report_sections.append(table_to_markdown(top_offenses, 30))

report_sections.append("\n\n## Sensitive Field Handling\n")
report_sections.append(
    "Victim and suspect demographic columns are profiled for data quality only. "
    "They should be excluded from the first forecasting model to avoid person-level profiling and proxy-based unfairness.\n"
)
report_sections.append(table_to_markdown(sensitive_profile[["column_name", "missing_pct", "approx_unique_count"]], 20))

report_sections.append("\n\n## Recommended Next Actions\n")
report_sections.append("1. Normalize null-like values and invalid categories in a reproducible cleaning pipeline.\n")
report_sections.append("2. Use `CMPLNT_FR_DT` and `CMPLNT_FR_TM` to create a canonical incident timestamp.\n")
report_sections.append("3. Keep quality flags for invalid dates, long durations, missing boroughs, missing precincts, and invalid coordinates.\n")
report_sections.append("4. Use weekly and monthly aggregate parquet files for baseline forecasting and dashboard summaries.\n")
report_sections.append("5. Exclude suspect and victim demographic fields from the first prediction model; use them only for fairness and data coverage review.\n")

report_path = REPORTS_DIR / "data_quality_report.md"
with open(report_path, "w", encoding="utf-8") as file:
    file.write("\n".join(report_sections))

print(f"Wrote data quality report: {report_path.relative_to(PROJECT_ROOT).as_posix()}")


# 13. Final EDA Checklist

Use this checklist after running the notebook:

- Confirm row count and date range are plausible.
- Confirm high-missing columns are understood and not accidentally used in core model features.
- Review invalid date/time summaries and decide cleaning rules.
- Review coordinate quality and decide whether to filter invalid coordinates or retain them with flags.
- Confirm top borough, precinct, and offense distributions match expectations.
- Review recent-change and rolling-deviation tables for early anomaly logic.
- Use generated weekly/monthly parquet files as the input to baseline forecasting.
- Keep demographic fields out of the first forecasting model.

Recommended next notebook or script:

- Build `data/processed/complaints_clean.parquet` with typed columns, normalized nulls, canonical timestamps, quality flags, and cleaned category labels.
- Build baseline forecasts using last-week, trailing 4-week mean, trailing 8-week weighted mean, and previous-year same-week baselines.


In [ ]:
print("EDA notebook run complete.")
print(f"Schema profile: {schema_profile_path.relative_to(PROJECT_ROOT).as_posix()}")
print(f"Dashboard summary: {dashboard_summary_path.relative_to(PROJECT_ROOT).as_posix()}")
print(f"Data quality report: {report_path.relative_to(PROJECT_ROOT).as_posix()}")
print(f"Weekly aggregate: {weekly_output_path.relative_to(PROJECT_ROOT).as_posix()}")
print(f"Monthly aggregate: {monthly_output_path.relative_to(PROJECT_ROOT).as_posix()}")
